# Этап 04. Реальные 24 месяца и рекомендуемый годовой контекст
Основной месячный view Сбера сохраняется как тензор 2016 × 24 × 6 и матрица 2016 × 144.
Рекомендованные БДМО формируются с учётом масштаба единиц и правильного знаменателя. Пропуски сохраняются. Основной годовой набор допускает только признаки с ≥90% покрытия в каждом году **и в паре лет**.
Уровни двух лет и межгодовые разности сохраняются отдельно, чтобы не утроить вес одной величины при кластеризации.


In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.bdmo_workflow import IndicatorStore, base_panel, load_mapping
from src.recommendations import read_plan, recommended_queue, search_catalog
config = json.loads((ROOT / 'configs/bdmo.json').read_text(encoding='utf-8'))
OUTPUT_DIR = ROOT / config['output_dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
panel, base_ids = base_panel(ROOT / config['base_panel'], config['expected_municipalities'], **config['calendar'])
mapping = load_mapping(ROOT / config['mapping_path'], base_ids, config['history_years'])
store = IndicatorStore(ROOT, config)
plan = read_plan(ROOT, config)
pd.set_option('display.max_colwidth', 120)
print(f'База: {len(base_ids)} МО × 24 месяца')


База: 2016 МО × 24 месяца


In [2]:
from src.clustering import prepare_dynamics, preprocess_context
from src.recommended_features import build_recommended_features, flatten_annual, redundancy_report
FEATURE_DIR = OUTPUT_DIR / 'features'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
X_dynamic = prepare_dynamics(panel, FEATURE_DIR)
data = pd.read_parquet(OUTPUT_DIR / 'bdmo_long.parquet')
levels, core, feature_coverage, dictionary = build_recommended_features(
    data, mapping, base_ids, plan, config['focus_years'], config['core_min_coverage'])
feature_coverage.to_csv(FEATURE_DIR / 'feature_coverage.csv', index=False)
dictionary.to_csv(FEATURE_DIR / 'feature_dictionary.csv', index=False)
redundancy_report(levels).to_csv(FEATURE_DIR / 'redundancy_2024.csv', index=False)
display(feature_coverage)


,feature,paired_coverage,coverage_2023,coverage_2024,minimum_coverage,core_eligible
0,wage,0.992560,1.000000,0.992560,0.992560,True
1,employment_per_1000,0.914683,0.958829,0.917659,0.917659,True
2,reporting_companies_per_1000,0.993552,0.997024,0.995536,0.995536,True
3,profitable_share,0.992063,1.000000,0.992063,0.992063,True
4,investment_total_per_capita,0.909722,0.955357,0.912698,0.912698,True
5,shop_floor_per_capita,0.915179,0.930556,0.935516,0.930556,True
6,population_density,0.987103,0.997024,0.990079,0.990079,True
7,lit_street_share,0.939980,0.949405,0.939980,0.939980,True


In [3]:
X_annual, X_delta = flatten_annual(core, base_ids, config['focus_years'])
assert X_annual.index.equals(base_ids)
X_annual.to_parquet(FEATURE_DIR / 'X_annual_core_raw.parquet')
X_annual.isna().to_parquet(FEATURE_DIR / 'X_annual_core_missing.parquet')
X_delta.to_parquet(FEATURE_DIR / 'X_annual_core_delta.parquet')
levels.to_parquet(FEATURE_DIR / 'annual_all_candidates.parquet')
print('Динамика:', X_dynamic.shape, 'Годовой контекст:', X_annual.shape)
print('Доля МО без пропусков во всём годовом контексте:', X_annual.notna().all(axis=1).mean())


Динамика: (2016, 144) Годовой контекст: (2016, 16)
Доля МО без пропусков во всём годовом контексте: 0.8303571428571429


Для основной модели используем месячную динамику Сбера. 

In [4]:
IMPUTE_ANNUAL = False
if IMPUTE_ANNUAL:
    prepared, missing_mask, parameters = preprocess_context(X_annual, impute=True)
    prepared.to_parquet(FEATURE_DIR / 'X_annual_core_prepared.parquet')
    parameters.to_csv(FEATURE_DIR / 'annual_preprocessing.csv')
